In [2]:
import os
import re
import glob
import numpy as np
import pandas as pd

In [ ]:

# ==========================================================
# INPUT / OUTPUT 
# ==========================================================
# Input comes from the smooth xyz data folder
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Smooth"
# Output goes to a sibling folder for orientation
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Body_Orientation"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:

# ==========================================================
# HELPERS
# ==========================================================
def normalize(v):
    n = np.linalg.norm(v, axis=1, keepdims=True)
    n[n == 0] = np.nan
    return v / n

def unwrap_nan(a):
    a = a.astype(float).copy()
    mask = ~np.isnan(a)
    # Convert to radians, unwrap, then back to degrees
    a[mask] = np.unwrap(np.radians(a[mask])) * 180 / np.pi
    return a

# ==========================================================
# FIND FILES
# ==========================================================
# Specifically looking for the _SMOOTH files
csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv")))

if not csv_files:
    raise FileNotFoundError(f"No SMOOTH CSV files found in {INPUT_DIR}")

# ==========================================================
# PROCESS
# ==========================================================
for path in csv_files:
    fname = os.path.basename(path)
    print(f"Calculating Orientation for: {fname}")

    df = pd.read_csv(path)
    t = df["frame"].values

    # Extract coordinates
    bead   = df[["pt1_X","pt1_Y","pt1_Z"]].values
    head   = df[["pt2_X","pt2_Y","pt2_Z"]].values
    lwb    = df[["pt3_X","pt3_Y","pt3_Z"]].values
    rwb    = df[["pt4_X","pt4_Y","pt4_Z"]].values
    center = df[["center_X","center_Y","center_Z"]].values

    # ======================================================
    # CENTER-BASED ORIENTATION (Body-centric)
    # ======================================================
    y_axis_c = normalize(head - center)      # forward
    x_axis_c = normalize(lwb - rwb)           # left-right
    z_axis_c = normalize(np.cross(x_axis_c, y_axis_c))
    x_axis_c = normalize(np.cross(y_axis_c, z_axis_c))  # re-orthogonalize

    yaw_c = unwrap_nan(np.degrees(np.arctan2(y_axis_c[:,1], y_axis_c[:,0])))
    pitch_c = unwrap_nan(np.degrees(
        np.arctan2(
            y_axis_c[:,2],
            np.sqrt(y_axis_c[:,0]**2 + y_axis_c[:,1]**2)
        )
    ))
    roll_c = unwrap_nan(np.degrees(
        np.arctan2(-x_axis_c[:,2], z_axis_c[:,2])
    ))

    # ======================================================
    # HEAD-BASED ORIENTATION (Gaze/Target-centric)
    # ======================================================
    y_axis_h = normalize(bead - head)         # forward towards target
    x_axis_h = normalize(lwb - rwb)           # left-right
    z_axis_h = normalize(np.cross(x_axis_h, y_axis_h))
    x_axis_h = normalize(np.cross(y_axis_h, z_axis_h))  # re-orthogonalize

    yaw_h = unwrap_nan(np.degrees(np.arctan2(y_axis_h[:,1], y_axis_h[:,0])))
    pitch_h = unwrap_nan(np.degrees(
        np.arctan2(
            y_axis_h[:,2],
            np.sqrt(y_axis_h[:,0]**2 + y_axis_h[:,1]**2)
        )
    ))
    roll_h = unwrap_nan(np.degrees(
        np.arctan2(-x_axis_h[:,2], z_axis_h[:,2])
    ))

    # ======================================================
    # SAVE CSV
    # ======================================================
    out_df = pd.DataFrame({
        "frame": t,
        "yaw_center": yaw_c,
        "pitch_center": pitch_c,
        "roll_center": roll_c,
        "yaw_head": yaw_h,
        "pitch_head": pitch_h,
        "roll_head": roll_h
    })

    # Transform "TrialX_SMOOTH.csv" -> "TrialX_ORIENTATION.csv"
    out_name = fname.replace("_SMOOTH.csv", "_ORIENTATION.csv")
    out_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)

    print(f"Saved: {out_name}")

print(f"\nSuccess: Orientation CSVs saved to {OUTPUT_DIR}")

Calculating Orientation for: Trial2_180rpmxyzpts_SMOOTH.csv
Saved: Trial2_180rpmxyzpts_ORIENTATION.csv
Calculating Orientation for: Trial2_200rpmxyzpts_SMOOTH.csv
Saved: Trial2_200rpmxyzpts_ORIENTATION.csv
Calculating Orientation for: Trial3_180rpmxyzpts_SMOOTH.csv
Saved: Trial3_180rpmxyzpts_ORIENTATION.csv
Calculating Orientation for: Trial4_400rpmxyzpts_SMOOTH.csv
Saved: Trial4_400rpmxyzpts_ORIENTATION.csv
Calculating Orientation for: Trial5_180rpmxyzpts_SMOOTH.csv
Saved: Trial5_180rpmxyzpts_ORIENTATION.csv
Calculating Orientation for: Trial5_400rpmxyzpts_SMOOTH.csv
Saved: Trial5_400rpmxyzpts_ORIENTATION.csv
Calculating Orientation for: Trial7_400rpmxyzpts_SMOOTH.csv
Saved: Trial7_400rpmxyzpts_ORIENTATION.csv

Success: Orientation CSVs saved to ./../../dataFolders/MuscaChasingBeads/Body_Orientation
